# 04. Consistency-Aware Multimodal Training

This notebook trains the proposed consistency-aware multimodal fusion model.

**Model D**: Consistency-aware fusion with CLIP similarity features

Architecture:
```
Text (BERT) ──┐
             ├──> Project ──> Concatenate ──> MLP ──> Prediction
Image (ViT) ─┤                                              ▲
             └──> Project ──> |T-I| ──> T*I ────────────────┘
                       CLIP Sim ──────────────────────────────>
```

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').absolute()))

import torch
import numpy as np
import pandas as pd

from src.utils import set_seed, get_device, ensure_dir, load_config
from src.dataset import MultimodalFakeNewsDataset, Sample
from src.preprocessing import MultimodalPreprocessor
from src.classifier import MultimodalClassifier, ClassifierConfig
from src.train import Trainer, TrainingConfig
from src.metrics import compute_metrics, print_metrics
from src.plotting import plot_training_history, plot_validation_metrics
from torch.utils.data import DataLoader

## Configuration

In [ ]:
# Set random seed for reproducibility
set_seed(42)

# Get device
device = get_device()
print(f"Using device: {device}")

# Paths
DATA_DIR = Path('data/processed')
IMAGES_DIR = Path('data/fakeddit/images')
CHECKPOINT_DIR = Path('artifacts/checkpoints')
RESULTS_DIR = Path('results')
ensure_dir(CHECKPOINT_DIR)
ensure_dir(RESULTS_DIR)

## Load Dataset

In [ ]:
# Load data splits
train_df = pd.read_csv(DATA_DIR / 'train.csv')
val_df = pd.read_csv(DATA_DIR / 'val.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

In [ ]:
def create_samples(df, images_dir):
    samples = []
    for _, row in df.iterrows():
        text = str(row.get('clean_title', row.iloc[0]))
        image_id = str(row.get('image_id', ''))
        label = int(row.get('binary_label', row.iloc[-1]))
        
        image_path = images_dir / f"{image_id}.jpg" if image_id else None
        
        sample = Sample(
            sample_id=str(row.name),
            text=text,
            image_path=str(image_path) if image_path and image_path.exists() else None,
            label=label,
            metadata={}
        )
        samples.append(sample)
    return samples

train_samples = create_samples(train_df, IMAGES_DIR)
val_samples = create_samples(val_df, IMAGES_DIR)
test_samples = create_samples(test_df, IMAGES_DIR)

train_dataset = MultimodalFakeNewsDataset(train_samples)
val_dataset = MultimodalFakeNewsDataset(val_samples)
test_dataset = MultimodalFakeNewsDataset(test_samples)

print(f"Datasets: Train={len(train_dataset)}, Val={len(val_dataset)}, Test={len(test_dataset)}")

In [ ]:
# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=4)

print("Data loaders created")

## Initialize Preprocessor and Model

In [ ]:
# Initialize preprocessor
preprocessor = MultimodalPreprocessor(
    text_model_name='google-bert/bert-base-uncased',
    image_model_name='google/vit-base-patch16-224',
    clip_model_name='openai/clip-vit-base-patch32',
    max_text_length=128,
    device=device
)

print("Preprocessor initialized")

In [ ]:
# Create consistency-aware model (Model D)
print("="*50)
print("Training Model D: Consistency-Aware Multimodal Fusion")
print("="*50)

consistency_config = ClassifierConfig(
    model_type='consistency_aware',
    freeze_encoders=True,
    projection_dim=256,
    fusion_hidden_dim=256,
    dropout=0.3
)

consistency_model = MultimodalClassifier(consistency_config)
consistency_model.to(device)

print(f"Model type: {consistency_config.model_type}")
print(f"Total parameters: {sum(p.numel() for p in consistency_model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in consistency_model.parameters() if p.requires_grad):,}")

## Training Configuration

In [ ]:
# Training configuration
train_config = TrainingConfig(
    batch_size=16,
    epochs=5,
    learning_rate=1e-3,
    weight_decay=1e-4,
    gradient_clip_norm=1.0,
    early_stopping_patience=3,
    device=str(device),
    checkpoint_dir=str(CHECKPOINT_DIR / 'consistency_aware'),
    log_interval=10,
    save_best_only=True,
    seed=42
)

print("Training configuration:")
print(f"  Batch size: {train_config.batch_size}")
print(f"  Epochs: {train_config.epochs}")
print(f"  Learning rate: {train_config.learning_rate}")
print(f"  Weight decay: {train_config.weight_decay}")

## Train the Model

In [ ]:
# Create trainer
trainer = Trainer(
    model=consistency_model,
    preprocessor=preprocessor,
    config=train_config,
    train_loader=train_loader,
    val_loader=val_loader
)

# Train
print("Starting training...")
history = trainer.train()

## Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

# Plot loss curves
fig = plot_training_history(
    history.__dict__,
    output_path=str(RESULTS_DIR / 'consistency_aware_loss.png'),
    title="Consistency-Aware Model Training Loss"
)
plt.show()

In [ ]:
# Plot validation metrics
fig = plot_validation_metrics(
    history.val_metrics,
    output_path=str(RESULTS_DIR / 'consistency_aware_metrics.png'),
    title="Consistency-Aware Model Validation Metrics"
)
plt.show()

## Train Ablation Model (Model E: Without CLIP Similarity)

In [ ]:
print("="*50)
print("Training Model E: Ablation (Without CLIP Similarity)")
print("="*50)

# Create ablation model
ablation_config = ClassifierConfig(
    model_type='ablation',
    freeze_encoders=True,
    projection_dim=256,
    fusion_hidden_dim=256,
    dropout=0.3
)

ablation_model = MultimodalClassifier(ablation_config)
ablation_model.to(device)

print(f"Ablation model created")

In [ ]:
# Create trainer for ablation model
ablation_train_config = TrainingConfig(
    batch_size=16,
    epochs=5,
    learning_rate=1e-3,
    device=str(device),
    checkpoint_dir=str(CHECKPOINT_DIR / 'ablation'),
    seed=42
)

ablation_trainer = Trainer(
    model=ablation_model,
    preprocessor=preprocessor,
    config=ablation_train_config,
    train_loader=train_loader,
    val_loader=val_loader
)

# Train ablation model
ablation_history = ablation_trainer.train()

## Save All Results

In [ ]:
from src.utils import save_json

# Save training histories
all_results = {
    'consistency_aware': {
        'best_val_f1': history.best_val_f1,
        'best_epoch': history.best_epoch,
        'train_losses': history.train_losses,
        'val_losses': history.val_losses
    },
    'ablation': {
        'best_val_f1': ablation_history.best_val_f1,
        'best_epoch': ablation_history.best_epoch,
        'train_losses': ablation_history.train_losses,
        'val_losses': ablation_history.val_losses
    }
}

save_json(all_results, RESULTS_DIR / 'consistency_training_results.json')

print("Results saved to results/consistency_training_results.json")

In [ ]:
# Summary
print("\n" + "="*50)
print("CONSISTENCY-AWARE TRAINING COMPLETE")
print("="*50)
print(f"\nModel D (Consistency-Aware):")
print(f"  Best Val F1: {history.best_val_f1:.4f}")
print(f"  Best Epoch: {history.best_epoch + 1}")

print(f"\nModel E (Ablation):")
print(f"  Best Val F1: {ablation_history.best_val_f1:.4f}")
print(f"  Best Epoch: {ablation_history.best_epoch + 1}")

print(f"\nCheckpoints saved to:")
print(f"  {CHECKPOINT_DIR / 'consistency_aware'}")
print(f"  {CHECKPOINT_DIR / 'ablation'}")

print("\nNext: Run 05_image_swap_experiment.ipynb")